# Indian Election Data Analysis

Exploratory and statistical analysis of Indian state assembly (**Vidhan Sabha**) and
parliamentary (**Lok Sabha**) election results.

**Vidhan Sabha:** data cleaning, turnout, candidate participation, vote share vs seat share, Gujarat case study (seats, vote share, victory margins).
**Lok Sabha:** gender representation, party performance, vote concentration (Gini / Lorenz), skewness & kurtosis, ANOVA, t-test, correlations, incumbency retention and vote-share swing.

> Data is read from the public URLs in the configuration cell, so an internet connection is required.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_style("whitegrid")

## 1. Load data

In [ ]:
BASE_URL = "https://samatrix-data.s3.ap-south-1.amazonaws.com/Statistics-Project"
df_lok = pd.read_csv(f"{BASE_URL}/ind-lok-sabha.csv")
df_vidhan = pd.read_csv(f"{BASE_URL}/ind-vidhan-sabha.csv")

print("Vidhan Sabha:", df_vidhan.shape)
print("Lok Sabha:   ", df_lok.shape)
df_vidhan.head()

## 2. Data cleaning

### 2.1 Vidhan Sabha

In [ ]:
# Party abbreviation: fill gaps using the most frequent abbreviation seen for each party name,
# treat anything still missing as Independent, and merge the two Congress labels.
abbrev_by_party = (
    df_vidhan.dropna(subset=["partyname", "partyabbre"])
    .groupby("partyname")["partyabbre"]
    .agg(lambda x: x.value_counts().idxmax())
)
df_vidhan["partyabbre"] = (
    df_vidhan["partyabbre"]
    .fillna(df_vidhan["partyname"].map(abbrev_by_party))
    .fillna("IND")
    .replace({"INC(I)": "INC"})
)

df_vidhan["year"] = df_vidhan["year"].astype(int)
df_vidhan["cand_sex"] = df_vidhan["cand_sex"].fillna("U")      # U = unknown
df_vidhan = df_vidhan.dropna(subset=["totvotpoll"])

# Constituency name: fill gaps with the most frequent name for the same (state, ac_no)
ac_name_by_seat = (
    df_vidhan.dropna(subset=["ac_name"])
    .groupby(["st_name", "ac_no"])["ac_name"]
    .agg(lambda x: x.value_counts().idxmax())
)
seat_index = pd.MultiIndex.from_frame(df_vidhan[["st_name", "ac_no"]])
df_vidhan["ac_name"] = df_vidhan["ac_name"].fillna(
    pd.Series(ac_name_by_seat.reindex(seat_index).values, index=df_vidhan.index)
)

print("Missing ac_name after fill:", df_vidhan["ac_name"].isna().sum())

### 2.2 Lok Sabha

In [ ]:
df_lok["cand_sex"] = df_lok["cand_sex"].fillna("U")
df_lok["pc_type"] = df_lok["pc_type"].fillna("Unknown")
df_lok["partyabbre"] = df_lok["partyabbre"].fillna("IND").replace({"INC(I)": "INC"})
df_lok["totvotpoll"] = df_lok["totvotpoll"].fillna(0)

df_lok.isnull().sum()

### 2.3 Helpers

Winners are the highest-polling candidate in each constituency and election year.

In [ ]:
def winners_of(df, seat_cols):
    """Return one row per constituency-election: the candidate with the most votes."""
    return df.loc[df.groupby(seat_cols)["totvotpoll"].idxmax()]


def party_vote_share(df, by=("year",)):
    """Each party's share (%) of all votes polled within the given grouping (default: per year)."""
    by = list(by)
    votes = df.groupby(by + ["partyabbre"])["totvotpoll"].sum().rename("party_votes").reset_index()
    total = df.groupby(by)["totvotpoll"].sum().rename("total_votes").reset_index()
    out = votes.merge(total, on=by)
    out["vote_share_percent"] = out["party_votes"] / out["total_votes"] * 100
    return out


def top_n_parties(df, n, by="party_votes"):
    return df.groupby("partyabbre")[by].sum().nlargest(n).index.tolist()


vidhan_winners = winners_of(df_vidhan, ["year", "st_name", "ac_no"])
lok_winners = winners_of(df_lok, ["year", "st_name", "pc_no"])

## 3. Vidhan Sabha: overview

In [ ]:
print("States:             ", df_vidhan["st_name"].nunique())
print("Election years:     ", df_vidhan["year"].nunique())
print("Constituency numbers:", df_vidhan["ac_no"].nunique())
print("Unique candidates:  ", df_vidhan["cand_name"].nunique())
print("Unique parties:     ", df_vidhan["partyabbre"].nunique())

### 3.1 Candidate gender distribution

In [ ]:
gender_counts = df_vidhan["cand_sex"].value_counts()
print(gender_counts)

plt.figure(figsize=(6, 6))
plt.pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%",
        startangle=140, colors=sns.color_palette("pastel"))
plt.title("Candidate Gender Distribution (Vidhan Sabha)")
plt.tight_layout()
plt.show()

### 3.2 Average candidates per seat

In [ ]:
avg_candidates = (
    df_vidhan.groupby(["year", "st_name", "ac_no"])["cand_name"].nunique()
    .groupby("year").mean()
    .rename("avg_candidates_per_seat")
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=avg_candidates, x="year", y="avg_candidates_per_seat", marker="o")
plt.title("Average Candidates per Seat per Year (Vidhan Sabha)")
plt.xlabel("Year")
plt.ylabel("Avg candidates")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.3 Voter turnout

In [ ]:
# Total votes polled in a seat = sum over candidates; electors are repeated on every row, so take max.
seat_turnout = (
    df_vidhan.groupby(["year", "st_name", "ac_no"])
    .agg(votes=("totvotpoll", "sum"), electors=("electors", "max"))
    .reset_index()
)

yearly_turnout = seat_turnout.groupby("year")[["votes", "electors"]].sum()
yearly_turnout["turnout_percent"] = yearly_turnout["votes"] / yearly_turnout["electors"] * 100

plt.figure(figsize=(12, 5))
sns.lineplot(data=yearly_turnout.reset_index(), x="year", y="turnout_percent", marker="o", color="green")
plt.title("Voter Turnout by Year (Vidhan Sabha)")
plt.xlabel("Year")
plt.ylabel("Turnout (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
state_turnout = (
    seat_turnout.groupby("st_name")[["votes", "electors"]].sum()
    .assign(turnout_percent=lambda d: d["votes"] / d["electors"] * 100)["turnout_percent"]
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 8))
ax = sns.barplot(x=state_turnout.values, y=state_turnout.index, hue=state_turnout.index,
                 palette="viridis", legend=False)
for p in ax.patches:
    ax.annotate(f"{p.get_width():.1f}%", (p.get_width(), p.get_y() + p.get_height() / 2),
                ha="left", va="center", fontsize=9)
plt.title("Voter Turnout by State (Vidhan Sabha)")
plt.xlabel("Turnout (%)")
plt.ylabel("State")
plt.tight_layout()
plt.show()

### 3.4 Constituencies per state

In [ ]:
state_constituencies = df_vidhan.groupby("st_name")["ac_no"].nunique().sort_values(ascending=False)

plt.figure(figsize=(10, 8))
ax = sns.barplot(x=state_constituencies.values, y=state_constituencies.index,
                 hue=state_constituencies.index, palette="plasma", legend=False)
for p in ax.patches:
    ax.annotate(f"{int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height() / 2),
                ha="left", va="center", fontsize=9)
plt.title("Number of Constituencies per State (Vidhan Sabha)")
plt.xlabel("Constituencies")
plt.ylabel("State")
plt.tight_layout()
plt.show()

### 3.5 Top parties and vote share vs seat share

In [ ]:
top_by_candidates = df_vidhan["partyabbre"].value_counts().nlargest(10)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_by_candidates.values, y=top_by_candidates.index, hue=top_by_candidates.index,
            palette="viridis", legend=False)
plt.title("Top 10 Parties by Number of Candidates (Vidhan Sabha)")
plt.xlabel("Candidates")
plt.ylabel("Party")
plt.tight_layout()
plt.show()

In [ ]:
vote_share = (df_vidhan.groupby("partyabbre")["totvotpoll"].sum() / df_vidhan["totvotpoll"].sum() * 100)
seat_share = vidhan_winners["partyabbre"].value_counts(normalize=True) * 100

share_df = (
    pd.concat({"Vote share %": vote_share, "Seat share %": seat_share}, axis=1)
    .fillna(0)
    .nlargest(10, "Vote share %")
)

share_df.sort_values("Vote share %").plot.barh(figsize=(12, 6), color=["skyblue", "orange"])
plt.title("Vote Share vs Seat Share, Top 10 Parties (Vidhan Sabha, all states and years)")
plt.xlabel("Percentage")
plt.tight_layout()
plt.show()

## 4. Vidhan Sabha: Gujarat case study

In [ ]:
gujarat = df_vidhan[df_vidhan["st_name"] == "Gujarat"].copy()
gujarat_winners = vidhan_winners[vidhan_winners["st_name"] == "Gujarat"]

gujarat_winners.sort_values("year", ascending=False)[
    ["year", "ac_name", "cand_name", "partyabbre", "totvotpoll"]
].head(10)

In [ ]:
# Seats won by the top 3 parties, by year
top3_seats = gujarat_winners["partyabbre"].value_counts().nlargest(3).index
seats_by_year = (
    gujarat_winners[gujarat_winners["partyabbre"].isin(top3_seats)]
    .groupby(["year", "partyabbre"]).size().reset_index(name="seats")
)

plt.figure(figsize=(12, 5))
sns.barplot(data=seats_by_year, x="year", y="seats", hue="partyabbre")
plt.title("Seats Won by Top 3 Parties in Gujarat (Vidhan Sabha)")
plt.xlabel("Year")
plt.ylabel("Seats")
plt.xticks(rotation=45)
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Vote share of the top 3 parties (by total votes), by year
gj_share = party_vote_share(gujarat)
top3_votes = top_n_parties(gj_share, 3)

plt.figure(figsize=(12, 5))
sns.lineplot(data=gj_share[gj_share["partyabbre"].isin(top3_votes)],
             x="year", y="vote_share_percent", hue="partyabbre", marker="o")
plt.title("Vote Share of Top 3 Parties in Gujarat (Vidhan Sabha)")
plt.xlabel("Year")
plt.ylabel("Vote share (%)")
plt.xticks(rotation=45)
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.1 Victory margins

In [ ]:
gujarat["rank"] = gujarat.groupby(["year", "ac_no"])["totvotpoll"].rank(method="first", ascending=False)
first = gujarat[gujarat["rank"] == 1]
second = gujarat[gujarat["rank"] == 2]

margins = first.merge(second, on=["year", "ac_no"], suffixes=("_winner", "_runnerup"))
margins["vote_margin"] = margins["totvotpoll_winner"] - margins["totvotpoll_runnerup"]

top3_winners = first["partyabbre"].value_counts().nlargest(3).index
margins_top3 = margins[margins["partyabbre_winner"].isin(top3_winners)]

plt.figure(figsize=(10, 5))
sns.histplot(margins_top3["vote_margin"], bins=30, kde=True, color="navy")
plt.title("Distribution of Victory Margins (Top 3 Parties, Gujarat)")
plt.xlabel("Winner votes - runner-up votes")
plt.ylabel("Constituencies")
plt.tight_layout()
plt.show()

In [ ]:
avg_margin = (
    margins_top3.groupby(["year", "partyabbre_winner"])["vote_margin"].mean()
    .reset_index().rename(columns={"partyabbre_winner": "party"})
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=avg_margin, x="year", y="vote_margin", hue="party", marker="o")
plt.title("Average Victory Margin by Party (Top 3, Gujarat)")
plt.xlabel("Year")
plt.ylabel("Average margin (votes)")
plt.xticks(rotation=45)
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Year/party combinations outside the interquartile range of average margins
q1, q3 = avg_margin["vote_margin"].quantile([0.25, 0.75])
avg_margin[(avg_margin["vote_margin"] < q1) | (avg_margin["vote_margin"] > q3)].sort_values(
    "vote_margin", ascending=False
)

## 5. Lok Sabha: gender representation

In [ ]:
participation = df_lok["cand_sex"].value_counts().rename("Candidates")
wins = lok_winners["cand_sex"].value_counts().rename("Winners")

gender_analysis = pd.concat([participation, wins], axis=1).fillna(0)
gender_analysis["Win rate (%)"] = gender_analysis["Winners"] / gender_analysis["Candidates"] * 100
gender_analysis = gender_analysis.sort_values("Candidates", ascending=False)
gender_analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, pal in zip(axes, ["Candidates", "Winners", "Win rate (%)"],
                        ["Blues_d", "Greens_d", "Oranges_d"]):
    sns.barplot(x=gender_analysis.index, y=gender_analysis[col], hue=gender_analysis.index,
                palette=pal, legend=False, ax=ax)
    ax.set_title(f"{col} by Gender (Lok Sabha)")
    ax.set_xlabel("Gender")
plt.tight_layout()
plt.show()

## 6. Lok Sabha: party performance over time

In [ ]:
seats = lok_winners.groupby(["year", "partyabbre"]).size().reset_index(name="seats_won")
top5_seats = seats.groupby("partyabbre")["seats_won"].sum().nlargest(5).index

plt.figure(figsize=(12, 5))
sns.lineplot(data=seats[seats["partyabbre"].isin(top5_seats)],
             x="year", y="seats_won", hue="partyabbre", marker="o")
plt.title("Seats Won by Top 5 Parties (Lok Sabha)")
plt.xlabel("Election year")
plt.ylabel("Seats won")
plt.xticks(rotation=45)
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
lok_share = party_vote_share(df_lok)
top5 = top_n_parties(lok_share, 5)
top5_share = lok_share[lok_share["partyabbre"].isin(top5)]

plt.figure(figsize=(12, 5))
sns.lineplot(data=top5_share, x="year", y="vote_share_percent", hue="partyabbre", marker="o")
plt.title("National Vote Share of Top 5 Parties (Lok Sabha)")
plt.xlabel("Election year")
plt.ylabel("Vote share (%)")
plt.xticks(rotation=45)
plt.legend(title="Party", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Vote concentration: Gini coefficient & Lorenz curve

A Gini of 0 means every candidate polls the same number of votes; values near 1 mean votes are concentrated in very few candidates.

In [ ]:
def gini(values):
    """Gini coefficient of a non-negative 1-D array."""
    x = np.sort(np.asarray(values, dtype=float))
    n = len(x)
    return 2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum()) - (n + 1) / n


print(f"Overall Gini (all Lok Sabha candidates): {gini(df_lok['totvotpoll']):.3f}")

gini_by_year = df_lok.groupby("year")["totvotpoll"].apply(gini)

plt.figure(figsize=(10, 5))
plt.plot(gini_by_year.index, gini_by_year.values, marker="o")
plt.title("Gini Coefficient of Vote Distribution by Year (Lok Sabha)")
plt.xlabel("Year")
plt.ylabel("Gini coefficient")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
votes = np.sort(df_lok.loc[df_lok["totvotpoll"] > 0, "totvotpoll"].values)
cum_votes = np.cumsum(votes) / votes.sum()
cum_candidates = np.arange(1, len(votes) + 1) / len(votes)

plt.figure(figsize=(7, 6))
plt.plot(cum_candidates, cum_votes, label="Lorenz curve", color="darkblue")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Line of equality")
plt.title("Lorenz Curve of Vote Distribution (Lok Sabha)")
plt.xlabel("Cumulative share of candidates")
plt.ylabel("Cumulative share of votes")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Skewness & kurtosis

Candidate-level vote share is measured as the share of **registered electors** (`totvotpoll / electors`).

In [ ]:
df_lok["electors"] = df_lok["electors"].replace(0, np.nan)
df_lok["elector_share_percent"] = df_lok["totvotpoll"] / df_lok["electors"] * 100
elector_share = df_lok.loc[df_lok["elector_share_percent"].between(0, 100), "elector_share_percent"]

print(f"Skewness: {stats.skew(elector_share):.4f}")
print(f"Excess kurtosis: {stats.kurtosis(elector_share):.4f}")

df_lok[df_lok["elector_share_percent"].between(0, 100)].groupby("year")["elector_share_percent"].agg(
    Skewness=stats.skew, Kurtosis=stats.kurtosis
)

In [ ]:
# Party-level: distribution of each party's national vote share across elections
# (only parties that contested at least 3 elections)
n_elections = lok_share["partyabbre"].value_counts()
eligible = n_elections[n_elections >= 3].index

party_moments = (
    lok_share[lok_share["partyabbre"].isin(eligible)]
    .groupby("partyabbre")["vote_share_percent"]
    .agg(Skewness=stats.skew, Kurtosis=stats.kurtosis)
    .sort_values("Skewness", ascending=False)
    .dropna()
)

print("Most right-skewed parties:")
print(party_moments.head())
print("\nMost left-skewed parties:")
print(party_moments.tail())

In [ ]:
plot_data = pd.concat([party_moments.head(10), party_moments.tail(10)]).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, pal in zip(axes, ["Skewness", "Kurtosis"], ["coolwarm", "magma"]):
    sns.barplot(data=plot_data, x=col, y="partyabbre", hue="partyabbre", palette=pal,
                legend=False, ax=ax)
    ax.set_title(f"Party-wise Vote Share {col}")
    ax.set_ylabel("Party")
plt.tight_layout()
plt.show()

## 9. Hypothesis tests

### 9.1 One-way ANOVA: does a party's vote share differ across states?

H0: the mean state-level vote share of the chosen party is equal across states
(states with fewer than 3 elections are excluded).

In [ ]:
TARGET_PARTY = "BJP"   # try "INC", "AAP", ...

state_share = party_vote_share(df_lok, by=("year", "st_name"))
state_share = state_share[state_share["partyabbre"] == TARGET_PARTY]

counts = state_share["st_name"].value_counts()
state_share = state_share[state_share["st_name"].isin(counts[counts >= 3].index)]

groups = [g["vote_share_percent"].values for _, g in state_share.groupby("st_name")]
f_stat, p_value = stats.f_oneway(*groups)

print(f"F = {f_stat:.3f}, p = {p_value:.3e}")
print(f"Reject H0: {TARGET_PARTY}'s vote share differs significantly across states."
      if p_value < 0.05 else "Fail to reject H0: no significant difference across states.")

plt.figure(figsize=(14, 5))
sns.boxplot(data=state_share, x="st_name", y="vote_share_percent")
plt.title(f"{TARGET_PARTY} Vote Share Across States (Lok Sabha)")
plt.xlabel("State")
plt.ylabel("Vote share (%)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

### 9.2 Welch t-test: BJP vs. major opposition parties

Compares national vote shares (one observation per party per election) of BJP against the pooled
shares of INC, DMK, SP, BSP and AITC.
Note that observations from the same party across elections are not independent, so treat the p-value as indicative.

In [ ]:
FOCUS_PARTY = "BJP"
OPPOSITION = ["INC", "DMK", "SP", "BSP", "AITC"]

focus = lok_share[lok_share["partyabbre"] == FOCUS_PARTY]["vote_share_percent"]
opposition = lok_share[lok_share["partyabbre"].isin(OPPOSITION)]["vote_share_percent"]

t_stat, p_value = stats.ttest_ind(focus, opposition, equal_var=False)
print(f"t = {t_stat:.3f}, p = {p_value:.3e}")
print("Significant difference in mean vote share." if p_value < 0.05 else "No significant difference.")

compare = lok_share[lok_share["partyabbre"].isin([FOCUS_PARTY] + OPPOSITION)].assign(
    group=lambda d: np.where(d["partyabbre"] == FOCUS_PARTY, FOCUS_PARTY, "Opposition")
)
plt.figure(figsize=(7, 5))
sns.boxplot(data=compare, x="group", y="vote_share_percent", hue="group", palette="Set2", legend=False)
plt.title(f"Vote Share: {FOCUS_PARTY} vs. Opposition Parties (Lok Sabha)")
plt.xlabel("")
plt.ylabel("Vote share (%)")
plt.tight_layout()
plt.show()

## 10. Correlation of vote shares (top 5 parties)

In [ ]:
pivot = top5_share.pivot(index="year", columns="partyabbre", values="vote_share_percent").fillna(0)

plt.figure(figsize=(7, 6))
sns.heatmap(pivot.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation of National Vote Share (Top 5 Parties, Lok Sabha)")
plt.tight_layout()
plt.show()

## 11. Incumbency retention rate

Share of seats where the winning party matches the party that won the same constituency
(`st_name`, `pc_no`) at the previous election. The first election for each seat is excluded.
Constituency boundaries change over time (delimitation), so treat this as an approximation.

In [ ]:
w = lok_winners.sort_values(["st_name", "pc_no", "year"]).copy()
w["prev_party"] = w.groupby(["st_name", "pc_no"])["partyabbre"].shift()
w = w.dropna(subset=["prev_party"])
w["retained"] = w["partyabbre"] == w["prev_party"]

incumbency = w.groupby("year")["retained"].mean().mul(100).reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=incumbency, x="year", y="retained", marker="o")
plt.title("Incumbent Party Retention Rate (Lok Sabha)")
plt.xlabel("Election year")
plt.ylabel("Seats retained (%)")
plt.grid(True)
plt.tight_layout()
plt.show()

## 12. Vote share swing (top 5 parties)

In [ ]:
swing = top5_share.sort_values(["partyabbre", "year"]).copy()
swing["swing"] = swing.groupby("partyabbre")["vote_share_percent"].diff()

plt.figure(figsize=(12, 5))
sns.lineplot(data=swing.dropna(subset=["swing"]), x="year", y="swing", hue="partyabbre", marker="o")
plt.axhline(0, color="gray", linestyle="--")
plt.title("Vote Share Swing Between Consecutive Elections (Top 5 Parties, Lok Sabha)")
plt.xlabel("Year")
plt.ylabel("Swing (percentage points)")
plt.grid(True)
plt.tight_layout()
plt.show()